# 18.3 马尔可夫链蒙特卡洛 / MCMC (Metropolis-Hastings & Gibbs)

**中文**：18.1 靠共轭得到闭式后验，18.2 用高斯近似后验。但当后验**又复杂又没有闭式、维度还高**时怎么办？**MCMC(马尔可夫链蒙特卡洛)** 给出一个惊人巧妙的答案:*"既然写不出后验的公式，那就从后验里**抽样**——抽够多的样本，用它们的直方图代表后验。"* 更妙的是,MCMC **只需要能算未归一化的后验(似然×先验)**,完全**绕开那个最难算的归一化常数(证据)**。MCMC 是贝叶斯统计的主力引擎(PyMC/Stan 背后就是它)。
**English**: 18.1 used conjugacy for a closed-form posterior; 18.2 approximated it with a Gaussian. But what if the posterior is **complex, has no closed form, and is high-dimensional**? **MCMC (Markov Chain Monte Carlo)** gives a strikingly clever answer: *"if you can't write the posterior's formula, **draw samples** from it — enough samples, and their histogram represents the posterior."* Even better, MCMC **only needs the unnormalized posterior (likelihood × prior)**, entirely **bypassing the hardest-to-compute normalizing constant (the evidence)**. MCMC is the workhorse engine of Bayesian statistics (behind PyMC/Stan).

---

**中文**：核心思想:构造一条**马尔可夫链**(一串随机游走的样本),让它的**平稳分布(stationary distribution)恰好等于目标后验**。走够久之后,链上的样本就等价于从后验里抽的样本。最经典的两个算法:
**English**: Core idea: construct a **Markov chain** (a sequence of random-walk samples) whose **stationary distribution is exactly the target posterior**. After running long enough, the chain's samples are equivalent to samples from the posterior. Two classic algorithms:

**中文**：
**① Metropolis-Hastings (MH)**:从当前点提议一个新点，按一个精心设计的概率决定"接受"还是"拒绝":
**English**:
**① Metropolis-Hastings (MH)**: propose a new point from the current one, and accept/reject with a carefully designed probability:

$$\alpha=\min\Big(1,\ \frac{p(\theta'|\mathcal D)}{p(\theta|\mathcal D)}\Big)=\min\Big(1,\ \frac{\tilde p(\theta')}{\tilde p(\theta)}\Big)$$

**中文**：关键:接受率只用到后验的**比值**，归一化常数(证据 $p(\mathcal D)$)在分子分母上下一约就**没了**！所以只要能算**未归一化后验** $\tilde p=$ 似然×先验就够了。"往高处走一定接受，往低处走按比例概率接受"——这个规则保证链最终在后验高处停留得多、低处少，采样密度正比于后验。
**English**: Key: the acceptance ratio only uses the posterior's **ratio**, so the normalizing constant (evidence $p(\mathcal D)$) **cancels**! You only need the **unnormalized posterior** $\tilde p=$ likelihood × prior. "Always accept moves uphill, accept downhill moves with proportional probability" — this rule guarantees the chain spends time in high-posterior regions proportionally, so sample density ∝ posterior.

**中文**：
**② Gibbs 采样**:MH 的特例。当每个变量的**全条件分布**(固定其他变量时该变量的分布)好抽样时，就轮流从各自的全条件里直接抽——**无需拒绝，每步都接受**。适合高维、变量间有条件共轭结构的模型。
**English**:
**② Gibbs sampling**: a special case of MH. When each variable's **full conditional** (its distribution given all others fixed) is easy to sample, cycle through sampling each from its full conditional — **no rejection, every step accepted**. Ideal for high-dimensional models with conditional-conjugate structure.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 贝叶斯计算核心必考）**
> **中文**：MCMC=构造平稳分布为目标后验的马氏链, 从中采样。**MH**:提议+按后验比值接受/拒绝——**只需未归一化后验(证据约掉)**, 这是它的核心威力。**Gibbs**:轮流从全条件采样, 无拒绝(MH特例)。**关键实践**:①**burn-in(预热)** 丢掉开头未收敛的样本;②**proposal 步长**要调——太小→接受率高但挪得慢(高自相关)、太大→老被拒(卡住), 甜点接受率约 **0.2~0.5**;③**诊断**:trace plot(看混合)、自相关/**有效样本量 ESS**、多链 **R-hat(≈1 才收敛)**。**vs 变分**:MCMC **渐近精确**但慢; VI 快但有偏。现代默认 **HMC/NUTS**(Stan/PyMC, 用梯度高效探索高维)。
> **English**: MCMC = build a Markov chain whose stationary distribution is the target posterior, then sample. **MH**: propose + accept/reject by the posterior ratio — **needs only the unnormalized posterior (evidence cancels)**, its core power. **Gibbs**: cycle sampling from full conditionals, no rejection (a special case of MH). **Key practice**: ① **burn-in** — discard the un-converged start; ② tune the **proposal step** — too small → high acceptance but slow movement (high autocorrelation), too large → mostly rejected (stuck), sweet-spot acceptance ≈ **0.2–0.5**; ③ **diagnostics**: trace plots (mixing), autocorrelation / **effective sample size (ESS)**, multi-chain **R-hat (≈1 means converged)**. **vs variational**: MCMC is **asymptotically exact** but slow; VI is fast but biased. Modern default **HMC/NUTS** (Stan/PyMC, gradient-based efficient exploration).


In [ ]:

# ============================================================
# Metropolis-Hastings 在 Beta-Binomial 上 / MH on Beta-Binomial
# 中文:抛一枚偏心硬币 N 次, 观测正面数。用 Beta(a,b) 先验推断"正面概率 p"。
#      这个模型是共轭的——真后验= Beta(a+正面, b+反面) 我们【已知】, 好用来【验证 MH 采样对不对】。
# English: flip a biased coin N times, observe heads. Infer "heads prob p" with a Beta(a,b) prior.
#      This model is conjugate — the true posterior = Beta(a+heads, b+tails) is KNOWN, letting us VERIFY MH.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
np.random.seed(0)
true_p=0.7; N=50; heads=int(np.random.binomial(N,true_p)); tails=N-heads
a0,b0=2,2                                                    # 先验 Beta(2,2) / prior
def log_unnorm_post(p):                                      # 未归一化对数后验 = 对数先验 + 对数似然
    if p<=0 or p>=1: return -np.inf
    return (a0-1)*np.log(p)+(b0-1)*np.log(1-p) + heads*np.log(p)+tails*np.log(1-p)

def metropolis(steps=30000, prop_sd=0.1, seed=0):
    rng=np.random.default_rng(seed); p=0.5; lp=log_unnorm_post(p)
    samples=np.empty(steps); accepts=0
    for i in range(steps):
        p_new=p+rng.normal(0,prop_sd)                       # 提议:当前点加高斯扰动 / random-walk proposal
        lp_new=log_unnorm_post(p_new)
        if np.log(rng.random()) < lp_new-lp:                # 接受概率 min(1, ratio) 的对数形式 / accept?
            p, lp = p_new, lp_new; accepts+=1
        samples[i]=p
    return samples, accepts/steps

samples, acc = metropolis(prop_sd=0.1)
burn=3000; post=samples[burn:]                              # 丢弃预热 / discard burn-in
true_post=stats.beta(a0+heads, b0+tails)                    # 已知真后验(共轭)/ known true posterior
print(f"观测:{heads} 正 {tails} 反 / observed {heads} heads, {tails} tails, 接受率 accept={acc:.2f}")
print(f"MH 后验均值 {post.mean():.3f} 标准差 {post.std():.3f}")
print(f"真后验均值 {true_post.mean():.3f} 标准差 {true_post.std():.3f}  → MH 恢复了真后验! / MH matches truth")


**中文**：MH 采样几乎完美还原了已知的真后验——这验证了算法正确。但 MCMC 好不好用,关键在**提议步长(proposal step)** 的调节。太小,链挪得太慢(样本高度相关、探索不充分);太大,提议老被拒绝(链卡在原地)。下面对比三种步长的 **trace(轨迹图)** 和**接受率**。
**English**: MH samples almost perfectly reproduce the known true posterior — verifying correctness. But MCMC's usability hinges on tuning the **proposal step**. Too small, the chain moves too slowly (highly correlated samples, poor exploration); too large, proposals are mostly rejected (the chain stalls). Below we compare **trace plots** and **acceptance rates** for three step sizes.


In [ ]:

# ============================================================
# 提议步长的影响 / effect of proposal step size
# ============================================================
configs=[(0.01,"太小 too small"),(0.1,"合适 good"),(0.6,"太大 too large")]
traces={}
for sd,name in configs:
    s,a=metropolis(steps=30000,prop_sd=sd); traces[name]=(s,a)
    print(f"步长 prop_sd={sd:<5} {name:<16} 接受率 accept={a:.2f}")
def autocorr(x, maxlag=50):                                 # 自相关(衡量样本独立性)/ autocorrelation
    x=x-x.mean(); ac=np.correlate(x,x,"full")[len(x)-1:]; return (ac/ac[0])[:maxlag]
print("\n接受率约 0.2~0.5 时混合最好 / mixing is best when acceptance ≈ 0.2-0.5")


**中文**：现在演示 **Gibbs 采样**。用一个二维相关高斯(相关系数 $\rho=0.8$)——它的两个**全条件分布都是一维高斯**(已知),所以能轮流精确抽样、无需拒绝。看 Gibbs 如何"横一步竖一步"地填满这个斜椭圆。
**English**: Now **Gibbs sampling**. Use a 2D correlated Gaussian (correlation $\rho=0.8$) — both **full conditionals are 1D Gaussians** (known), so we sample each exactly, no rejection. Watch Gibbs fill the tilted ellipse with "horizontal then vertical" steps.


In [ ]:

# ============================================================
# Gibbs 采样二维相关高斯 / Gibbs sampling a correlated 2D Gaussian
# 中文:目标 N(0, [[1,ρ],[ρ,1]])。全条件: x|y ~ N(ρy, 1-ρ²), y|x ~ N(ρx, 1-ρ²)。
# English: target N(0, [[1,ρ],[ρ,1]]). Full conditionals: x|y ~ N(ρy, 1-ρ²), y|x ~ N(ρx, 1-ρ²).
# ============================================================
rho=0.8
def gibbs(steps=5000, seed=0):
    rng=np.random.default_rng(seed); x=y=0.0; out=np.empty((steps,2))
    sd=np.sqrt(1-rho**2)
    for i in range(steps):
        x=rng.normal(rho*y, sd)                             # 从 x 的全条件抽样 / sample x | y
        y=rng.normal(rho*x, sd)                             # 从 y 的全条件抽样 / sample y | x
        out[i]=(x,y)
    return out
G=gibbs(); Gb=G[500:]                                       # 丢预热 / burn-in
print(f"Gibbs 恢复的相关系数 / recovered correlation: {np.corrcoef(Gb.T)[0,1]:.3f} (真值 true {rho})")
print(f"边际均值 marginal means: {Gb.mean(0).round(3)} (真值 [0,0]), 标准差 std: {Gb.std(0).round(3)} (真值 [1,1])")


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(2,2,figsize=(14,9))
# ① MH 后验直方图 vs 真后验 / MH histogram vs true posterior
ax[0,0].hist(post,bins=50,density=True,alpha=0.6,color="#4C72B0",label="MH 采样")
xx=np.linspace(0.4,0.9,200); ax[0,0].plot(xx,true_post.pdf(xx),"r",lw=2,label="真后验 true Beta")
ax[0,0].axvline(true_p,color="g",ls="--",label="真值 true p"); ax[0,0].set_title("MH 恢复真后验 / MH recovers posterior"); ax[0,0].legend(fontsize=8); ax[0,0].set_xlabel("p")
# ② 三种步长的 trace / traces for 3 step sizes
for (sd,name),c in zip(configs,["#C44E52","#55A868","#DD8452"]):
    ax[0,1].plot(traces[name][0][:800],c,alpha=0.8,label=f"{name} (acc={traces[name][1]:.2f})")
ax[0,1].set_title("Trace 轨迹:步长决定混合 / traces (step controls mixing)"); ax[0,1].set_xlabel("iteration"); ax[0,1].set_ylabel("p"); ax[0,1].legend(fontsize=7)
# ③ 自相关:好步长衰减最快 / autocorrelation
for (sd,name),c in zip(configs,["#C44E52","#55A868","#DD8452"]):
    ax[1,0].plot(autocorr(traces[name][0][3000:]),c,label=name)
ax[1,0].axhline(0,color="k",lw=0.5); ax[1,0].set_title("自相关:衰减越快=有效样本越多 / autocorrelation"); ax[1,0].set_xlabel("lag"); ax[1,0].legend(fontsize=7)
# ④ Gibbs 探索二维高斯 / Gibbs exploring the 2D Gaussian
ax[1,1].plot(G[:80,0],G[:80,1],"o-",color="gray",alpha=0.5,ms=3,label="前80步路径")
ax[1,1].scatter(Gb[:,0],Gb[:,1],s=3,alpha=0.2,color="#4C72B0")
ax[1,1].set_title(f"Gibbs 采样相关高斯(ρ={rho}) / Gibbs on correlated Gaussian"); ax[1,1].set_xlabel("x"); ax[1,1].set_ylabel("y"); ax[1,1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/bay03_viz.png",dpi=80); plt.show()
print("步长太小→trace 挪得慢+自相关高; 太大→老被拒卡住; 合适→充分混合")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **MCMC 让"不可能"变可能**:我们**没有用**共轭闭式公式,纯靠"提议-接受/拒绝"的随机游走,就从后验里抽出样本、还原了真后验(直方图几乎完美贴合真 Beta 曲线)。核心魔法在接受率只用后验**比值**——那个最难算的归一化常数**约掉了**。这就是为什么 MCMC 能处理任意复杂、写不出归一化式子的后验。
2. **调步长是门手艺(trace + 自相关)**:三条 trace 一目了然——步长**太小**(红)时链像蜗牛,慢慢爬,样本高度自相关(有效样本少);**太大**(橙)时提议老被拒,链常卡在原地(平台);**合适**(绿)时充分"毛毛虫式"混合。经验法则:**调到接受率 0.2~0.5**。自相关图证实:好步长的自相关衰减最快=每个样本更"独立"、有效样本量最大。
3. **Gibbs 无拒绝但要能抽全条件**:Gibbs 每步都接受(横竖交替精确抽样),优雅高效,但**前提是全条件分布可抽样**(需要模型有条件共轭结构)。它天然适合层次贝叶斯模型(下节 18.5)。
4. **诚实的代价**:MCMC **渐近精确但慢**——要跑很多步、丢弃 burn-in、还得做收敛诊断(多链 R-hat、ESS)。高维/强相关后验混合极慢(随机游走 MH 尤其吃力),所以现代默认用带梯度的 **HMC/NUTS**(Stan/PyMC 的引擎),用后验的梯度信息高效穿越高维空间。

**English**:
1. **MCMC makes the "impossible" possible**: we used **no** conjugate closed form — just a "propose-accept/reject" random walk drew samples from the posterior and recovered it (the histogram matches the true Beta almost perfectly). The core magic is that the acceptance uses only the posterior **ratio** — the hardest normalizing constant **cancels**. This is why MCMC handles arbitrarily complex posteriors with no writable normalizer.
2. **Tuning the step is a craft (traces + autocorrelation)**: the three traces make it obvious — **too small** (red) and the chain crawls, highly autocorrelated (few effective samples); **too large** (orange) and proposals are mostly rejected, the chain often stalls (flat plateaus); **just right** (green) mixes well "caterpillar-like." Rule of thumb: **tune to acceptance 0.2–0.5**. The autocorrelation plot confirms: the good step's autocorrelation decays fastest = each sample is more "independent," maximizing effective sample size.
3. **Gibbs has no rejection but needs samplable conditionals**: Gibbs accepts every step (exact alternating sampling), elegant and efficient, but **requires the full conditionals to be samplable** (a conditional-conjugate structure). It naturally suits hierarchical Bayesian models (18.5 next).
4. **Honest costs**: MCMC is **asymptotically exact but slow** — many steps, discard burn-in, and run convergence diagnostics (multi-chain R-hat, ESS). High-dimensional / strongly-correlated posteriors mix very slowly (random-walk MH especially struggles), so the modern default is gradient-based **HMC/NUTS** (the engine of Stan/PyMC), which uses posterior gradients to traverse high-dim space efficiently.

> 💼 **实战视角 / Practical angle**
> **中文**:MCMC 是贝叶斯统计的**黄金标准**(要精确后验时用它)。落地:①**几乎不手写 MH**,用 **PyMC/Stan/NumPyro** 写模型,底层自动 NUTS 采样;②**必做诊断**:R-hat(<1.01)、ESS(足够大)、trace(目测混合)、能量图;③报告结果给**可信区间(credible interval)** 而非点估计;④慢是硬伤——大数据/复杂模型可能跑几小时,这时转向**变分推断(快, 18.4)**。面试金句:*"MCMC 构造平稳分布=后验的马氏链, MH 靠后验比值接受(约掉证据), Gibbs 靠全条件; 渐近精确但慢, 要调步长(接受率0.2~0.5)+诊断(R-hat/ESS); 现代用 HMC/NUTS。"*
> **English**: MCMC is the **gold standard** of Bayesian statistics (use it when you need an accurate posterior). In practice: ① **rarely hand-write MH** — use **PyMC/Stan/NumPyro**, which auto-run NUTS under the hood; ② **always diagnose**: R-hat (<1.01), ESS (large enough), trace (eyeball mixing), energy plots; ③ report **credible intervals**, not point estimates; ④ slowness is the pain — big data / complex models can take hours, then switch to **variational inference (fast, 18.4)**. Interview line: *"MCMC builds a Markov chain whose stationary distribution is the posterior; MH accepts by the posterior ratio (evidence cancels), Gibbs by full conditionals; asymptotically exact but slow, needing step tuning (acceptance 0.2-0.5) + diagnostics (R-hat/ESS); modern default is HMC/NUTS."*

---
### 小结 / Summary
- **中文**:MCMC=构造平稳分布为后验的马氏链采样; 只需未归一化后验(证据约掉)。
- **English**: MCMC = sample from a Markov chain whose stationary distribution is the posterior; needs only the unnormalized posterior (evidence cancels).
- **中文**:MH=提议+按后验比值接受; Gibbs=轮流从全条件采样(无拒绝); 都能还原真后验。
- **English**: MH = propose + accept by the posterior ratio; Gibbs = cycle full conditionals (no rejection); both recover the true posterior.
- **中文**:调步长(接受率0.2~0.5)+诊断(trace/自相关/R-hat/ESS)是关键; 渐近精确但慢→HMC/NUTS。
- **English**: Tuning the step (acceptance 0.2-0.5) + diagnostics (trace/autocorrelation/R-hat/ESS) are key; asymptotically exact but slow → HMC/NUTS.
